# Notebook for assignment Responsible AI predictive XAI exercise

In [ ]:
import os
import matplotlib.pyplot as plt
import torchvision
import torch
import pandas as pd
import numpy as np

from utils.notebook import display_scrollable_dataframe,plot_sailency
from data_loaders import CUB_extnded_dataset
from models import get_inception_transform
from IPython.display import display

from sailency import get_saliency_maps,saliency_score_part

In [ ]:
# Settings for the experiment you're running
data_set = 'val' 

# Number of images you want to collect for analysis
num_samples = 10  # Specify how many images you want to collect

# Settings for loading the dataset
data_config = {
    'CUB_dir': 'data/CUB_200_2011',
    'split_file': 'data/train_test_val.pkl',
    'use_majority_voting': True,
    'min_class_count': 10,
    'return_visibility': True
}


In [ ]:
import torch
import torchvision
import random
from torch.utils.data import DataLoader

# Define the data set and transformers
transformer = get_inception_transform(mode='val', methode="center")
human_transform = torchvision.transforms.Compose([
    torchvision.transforms.CenterCrop(299),
    torchvision.transforms.ToTensor()
])

# Load the dataset with the specified transformations
data = CUB_extnded_dataset(mode=data_set, config_dict=data_config, transform=transformer)
data_human = CUB_extnded_dataset(mode=data_set, config_dict=data_config, transform=human_transform)

# Extract concept and class names
concept_names = data.consept_labels_names
class_names = data.class_labels_names
n_classes = data.n_classes
n_concepts = data.n_concepts

# Automatically select multiple samples for analysis
num_samples = 100  # Specify the number of samples you want to analyze
sample_indices = random.sample(range(len(data)), num_samples)

# Filter both datasets based on the selected sample indices
filtered_data = [data[i] for i in sample_indices]
filtered_data_human = [data_human[i] for i in sample_indices]

# Create DataLoaders for both datasets
data_loader = DataLoader(filtered_data, batch_size=1, shuffle=False)
data_human_loader = DataLoader(filtered_data_human, batch_size=1, shuffle=False)

# Verify that both DataLoaders have the same samples
assert len(data_loader) == len(data_human_loader), "Mismatch in dataset lengths!"

# Print out details for confirmation
print(f"Loaded {len(filtered_data)} samples for analysis.")
print(f"Number of concepts: {n_concepts}")
print(f"Number of classes: {n_classes}")


In [ ]:
# Loop through each of the selected IDs from the previous step
for idx in sample_indices:
    # Get the data (normalized) for the selected index
    X, C, Y, coordinates = data.__getitem__(idx)
    
    # Print out shapes and details
    print(f"ID: {idx}")
    print("Image shape (normalized):", X.shape)
    print("Concept shape:", C.shape)
    print("Label shape:", Y.shape)
    print("Coordinates:", coordinates)
    
    # Add a batch dimension to the image tensor for model input
    X = X.unsqueeze(0)
    
    # Get the original (non-normalized) image for the same index
    img, _, _, _ = data_human.__getitem__(idx)
    
    # Print the label's class index
    print("Class label (argmax):", Y.argmax().item())
    print("-" * 50)  # Separator for readability


In [ ]:
# Loop through each of the selected IDs from the previous step
for idx in sample_indices:
    # Get the data (normalized) for the selected index
    X, C, Y, coordinates = data.__getitem__(idx)
    
    print(f"ID: {idx}")
    print("Image shape (normalized):", X.shape)
    print("Concept shape:", C.shape)
    print("Label shape:", Y.shape)
    print("Coordinates:", coordinates)
    
    # Add a batch dimension to the image tensor
    X = X.unsqueeze(0)
    
    # Get the original (non-normalized) image for the same index
    img, _, _, _ = data_human.__getitem__(idx)
    
    # Print the class label
    print("Class label (argmax):", Y.argmax().item())
    
    # Separate concepts and their visibility
    if len(C.shape) == 2:
        Concepts = C[0]
        Concepts_visiblity = C[1]
    else:
        Concepts = C.tolist()
        Concepts_visiblity = [None] * len(C)
    
    # Print details about concepts and visibility
    print("Concepts:", Concepts)
    print("Concepts Visibility:", Concepts_visiblity)
    print("-" * 50)  # Separator for readability


In [ ]:
# Loop through each of the selected IDs from the previous step
for idx in sample_indices:
    # Get the data (normalized) and original image for the selected index
    X, C, Y, coordinates = data.__getitem__(idx)
    img, _, _, _ = data_human.__getitem__(idx)
    
    # Add a batch dimension to the normalized image
    X = X.unsqueeze(0)
    
    # Separate concepts and their visibility
    if len(C.shape) == 2:
        Concepts = C[0]
        Concepts_visiblity = C[1]
    else:
        Concepts = C.tolist()
        Concepts_visiblity = [None] * len(C)
    
    # Display the image
    plt.figure(figsize=(7, 7))
    plt.imshow(img.permute(1, 2, 0))
    
    # Plot the concept coordinates
    for i in range(len(coordinates)):
        if len(coordinates[i]) > 0:
            for x, y in coordinates[i]:
                # Extract the concept name
                name = concept_names[i].split("_")[1]
                
                # Set offset for specific concept names
                if name == "forehead":
                    offset = -30
                elif name == "eye":
                    offset = 0
                else:
                    offset = 0
                
                # Annotate the concept name on the image
                plt.text(x + offset, y, name, fontsize=9, color='red')
    
    plt.axis('off')
    plt.title(f"ID: {idx}, Class: {Y.argmax().item()}")
    plt.show()
    print("-" * 50)  # Separator for readability


# Expandability for Sequential and independent models.  

In [ ]:
#Load models make sure the model is trained with the same settings as the data loader
model_folder = r"models/Sequential_Basemodel3"

X_to_C_path = os.path.join(model_folder,"best_XtoC_model.pth")
C_to_Y_path = os.path.join(model_folder,"best_CtoY_model.pth")

ModelXtoC = torch.load(X_to_C_path,map_location=torch.device('cpu'))
ModelCtoY = torch.load(C_to_Y_path,map_location=torch.device('cpu'))

In [ ]:
# Loop through each of the selected IDs from the previous step
for idx in sample_indices:
    # Get the data (normalized) for the selected index
    X, C, Y, coordinates = data.__getitem__(idx)
    X = X.unsqueeze(0).to(device)  # Add batch dimension and move to device

    # Make predictions using your models
    C_hat = ModelXtoC(X)
    Y_hat = ModelCtoY(C_hat)
    
    # Print the prediction results
    print(f"ID: {idx}")
    print(f"Predicted class: {Y_hat.argmax().item()}, True class: {Y.argmax().item()}")
    print(f"Probability of true class: {Y_hat[0, Y.argmax()].item():.4f}")
    print("-" * 50)  # Separator for readability


In [ ]:
import pandas as pd
import numpy as np

# Loop through each of the selected IDs from the previous step
for idx in sample_indices:
    # Get the data (normalized) for the selected index
    X, C, Y, coordinates = data.__getitem__(idx)
    X = X.unsqueeze(0).to(device)  # Add batch dimension and move to device

    # Make predictions using your models
    C_hat = ModelXtoC(X)
    Y_hat = ModelCtoY(C_hat)
    
    # Separate concepts and their visibility
    if len(C.shape) == 2:
        Concepts = C[0]
        Concepts_visiblity = C[1]
    else:
        Concepts = C.tolist()
        Concepts_visiblity = [None] * len(C)
    
    # Get the weights for the predicted class
    weights = ModelCtoY.linear.weight[Y.argmax().item()]  # Find the weights for the true class

    # Create a DataFrame to display the concept information
    Concept_frame = pd.DataFrame({
        "Concept": concept_names,
        "Concept_true": Concepts,
        "Concept_visiblity": Concepts_visiblity,
        "Concept_pred": np.round(C_hat[0].detach().numpy(), 2),
        "CtoY_weight": weights.detach().numpy()
    })
    
    # Display the DataFrame
    print(f"ID: {idx}")
    display_scrollable_dataframe(Concept_frame)
    print("-" * 50)  # Separator for readability


In [ ]:
# List of concepts to generate saliency maps for
concept_list = [51, 50, 111]  # You can adjust this list as needed

# Loop through each of the selected IDs from the previous step
for idx in sample_indices:
    # Get the data (normalized) and original image for the selected index
    X, C, Y, coordinates = data.__getitem__(idx)
    img, _, _, _ = data_human.__getitem__(idx)
    
    # Add a batch dimension and move to device
    X = X.unsqueeze(0).to(device)
    
    # Generate saliency maps for the selected concepts
    saliency_maps = get_saliency_maps(X, concept_list, ModelXtoC, method_type="vanilla")
    
    # Check if saliency maps were generated correctly
    if len(saliency_maps) < len(concept_list):
        print(f"Warning: Could not generate saliency maps for all concepts for ID {idx}.")
        print(f"Generated {len(saliency_maps)} out of {len(concept_list)} expected.")
    
    # Plot the saliency maps if they were generated correctly
    if len(saliency_maps) > 0:
        print(f"ID: {idx}")
        try:
            plot_sailency(img, saliency_maps, concept_list[:len(saliency_maps)], concept_names, coordinates)
        except IndexError as e:
            print(f"Error plotting saliency maps for ID {idx}: {e}")
        print("-" * 50)  # Separator for readability
    else:
        print(f"No saliency maps generated for ID {idx}.")


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# Initialize a list to store the summary for each sample
summary_data = []

# Loop through each of the selected IDs from the previous step
for idx in sample_indices:
    # Get the data (normalized) for the selected index
    X, C, Y, coordinates = data.__getitem__(idx)
    X = X.unsqueeze(0).to(device)  # Add batch dimension and move to device
    img, _, _, _ = data_human.__getitem__(idx)
    
    # Make predictions using your models
    C_hat = ModelXtoC(X)
    Y_hat = ModelCtoY(C_hat)
    
    # Separate concepts and their visibility
    if len(C.shape) == 2:
        Concepts = C[0]
        Concepts_visiblity = C[1]
    else:
        Concepts = C.tolist()
        Concepts_visiblity = [None] * len(C)
    
    # Extract weights for all concepts for the predicted class
    predicted_class_idx = Y_hat.argmax().item()
    weights = ModelCtoY.linear.weight[predicted_class_idx].detach().numpy()
    
    # Create a DataFrame to display the concept information
    Concept_frame = pd.DataFrame({
        "Concept": concept_names,
        "Concept_true": Concepts,
        "Concept_visiblity": Concepts_visiblity,
        "Concept_pred": np.round(C_hat[0].detach().numpy(), 2),
        "CtoY_weight": weights
    })
    
    # Filter to keep only the concepts where Concept_true == 1
    relevant_concepts = Concept_frame[Concept_frame["Concept_true"] == 1]

    # Check if there are any relevant concepts to work with
    if relevant_concepts.empty:
        continue

    # Generate saliency maps for all the relevant concepts
    relevant_indices = relevant_concepts.index.tolist()
    saliency_maps = get_saliency_maps(X, relevant_indices, ModelXtoC, method_type="vanilla")
    
    # Calculate the saliency score using saliency_score_part from saliency.py
    saliency_scores = []
    for i, concept_idx in enumerate(relevant_indices):
        concept_name = concept_names[concept_idx]
        saliency_score = 0  # Default saliency score
        
        # Get the visible concepts and their coordinates
        visible_idx, visible_coords = get_visible_consepts(coordinates)
        
        if concept_idx in visible_idx:
            coord_index = visible_idx.index(concept_idx)
            coordinates_for_concept = visible_coords[coord_index]
            
            if i < len(saliency_maps):
                # Calculate the saliency score using the provided function
                saliency_score = saliency_score_part(saliency_maps[i], coordinates_for_concept)
        
        # Append the saliency score to the DataFrame
        relevant_concepts.at[concept_idx, 'Saliency_score'] = saliency_score
    
    # Filter out concepts with zero saliency score
    relevant_concepts = relevant_concepts[relevant_concepts['Saliency_score'] > 0]
    
    # Select the top 3 concepts with the highest absolute CtoY_weight
    relevant_concepts['abs_weight'] = relevant_concepts['CtoY_weight'].abs()
    top_concepts = relevant_concepts.sort_values(by='abs_weight', ascending=False).head(3)
    
    # Store the selected top concepts in the summary
    for _, row in top_concepts.iterrows():
        summary_data.append({
            "ID": idx,
            "Concept": row["Concept"],
            "CtoY_weight": row["CtoY_weight"],
            "Concept_pred": row["Concept_pred"],
            "Class_pred": Y_hat.argmax().item(),
            "True_label": Y.argmax().item(),
            "Prediction_correct": Y_hat.argmax().item() == Y.argmax().item(),
            "Saliency_score": row["Saliency_score"]
        })

# Convert the summary data into a DataFrame
summary_df = pd.DataFrame(summary_data)

# Display the summary table using Jupyter's display functionality
display(summary_df)


In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Step 1: Calculate the absolute values of CtoY_weight for better analysis
summary_df['abs_CtoY_weight'] = summary_df['CtoY_weight'].abs()

# Step 2: Visualize the relationship using a scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(summary_df['abs_CtoY_weight'], summary_df['Saliency_score'], alpha=0.6)
plt.xlabel('Absolute CtoY_weight')
plt.ylabel('Saliency Score')
plt.title('Correlation between CtoY_weight and Saliency Score')
plt.grid(True)
plt.show()

# Step 3: Calculate the Pearson correlation coefficient
corr_coef, p_value = pearsonr(summary_df['abs_CtoY_weight'], summary_df['Saliency_score'])

# Step 4: Display the correlation results
print(f"Pearson Correlation Coefficient: {corr_coef:.3f}")
print(f"P-value: {p_value:.3f}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from IPython.display import display

# Step 1: Filter out rows where Saliency_score is zero
filtered_df = summary_df[summary_df["Saliency_score"] > 0]

# Step 2: Calculate the absolute values of CtoY_weight for better analysis
filtered_df['abs_CtoY_weight'] = filtered_df['CtoY_weight'].abs()

# Step 3: Visualize the relationship using a scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(filtered_df['abs_CtoY_weight'], filtered_df['Saliency_score'], alpha=0.6)
plt.xlabel('Absolute CtoY_weight')
plt.ylabel('Saliency Score')
plt.title('Correlation between CtoY_weight and Saliency Score')
plt.grid(True)
plt.show()

# Step 4: Calculate the Pearson correlation coefficient
corr_coef, p_value = pearsonr(filtered_df['abs_CtoY_weight'], filtered_df['Saliency_score'])

# Step 5: Display the correlation results
print(f"Pearson Correlation Coefficient: {corr_coef:.3f}")
print(f"P-value: {p_value:.3f}")

# Display a summary of the filtered data
display(filtered_df)


# Saliency map for Joint models  

In [ ]:
#Load models make sure the model is trained with the same settings as the data loader
model_path = r'models/Joint_Basemodel3/best_Joint_model.pth'
n_classes = data.n_classes
n_concepts = data.n_concepts

#Load the model
Joint_model = torch.load(model_path,map_location=torch.device('cpu'))




In [ ]:
#Make prediction
C_hat,Y_hat = Joint_model(X)
print(f"Predicted class: {Y_hat.argmax().item()} true class: {Y.argmax().item()} probability of true class: {Y_hat[0,Y.argmax()].item()}")

In [ ]:
#Make Concept prediction and other stuff
weights = Joint_model.MLP_model.linear.weight[Y.argmax().item()]
Concept_frame = pd.DataFrame({"Concept":concept_names,"Concept_true":Concepts,"Concept_visiblity":Concepts_visiblity,"Concept_pred":np.round(C_hat[0].detach().numpy(),2),"CtoY_weight":weights.detach().numpy()})
display_scrollable_dataframe(Concept_frame)

In [ ]:
concept_list = [51,50,111]

#Make model only predict the concept C
Joint_model.set_sailency_output("C")

sailency_maps = get_saliency_maps(X,concept_list,Joint_model,method_type="vanilla")

plot_sailency(img,sailency_maps,concept_list,concept_names,coordinates)

In [ ]:
concept_list = [62]

#Make model only predict the concept C
Joint_model.set_sailency_output("Y")

sailency_maps = get_saliency_maps(X,concept_list,Joint_model,method_type="vanilla")

plot_sailency(img,sailency_maps,concept_list,class_names)